In [1]:
import pandas as pd
import numpy as np
import json
import requests
import os
from dotenv import load_dotenv
from pathlib import Path
import time


Below is just checking output of API for 118th congress

In [68]:
load_dotenv()
root = "https://api.congress.gov"
endpoint = "/v3/bill/118"

url = root + endpoint
params = {"api_key": os.environ["CONGRESS_API_KEY"], "limit": 250, "offset": 0, "format": "json"}
resp = requests.get(url, params=params); resp.raise_for_status()


In [69]:
resp.json().keys()
resp.text

'{\n    "bills": [\n        {\n            "congress": 118,\n            "introducedDate": "2023-01-03",\n            "latestAction": {\n                "actionDate": "2023-01-03",\n                "text": "Submitted in the Senate, considered, and agreed to without amendment by Unanimous Consent. (consideration: CR S6; text: CR S6)"\n            },\n            "number": "1",\n            "originChamber": "Senate",\n            "originChamberCode": "S",\n            "title": "A resolution establishing a Committee to Inform the President of the United States that a quorum of each House is assembled.",\n            "type": "SRES",\n            "updateDate": "2024-04-17",\n            "updateDateIncludingText": "2024-04-17",\n            "url": "https://api.congress.gov/v3/bill/118/sres/1?format=json"\n        },\n        {\n            "congress": 118,\n            "introducedDate": "2023-01-03",\n            "latestAction": {\n                "actionDate": "2023-01-03",\n               

Keys are bills, pagination, and request.

In [70]:
#json_data = resp.json()
#json_data

# Configuration Stuff

In [71]:
CONGRESSES = [118] # trying one for now
BILL_TYPES = ["hr", "s", "hjres", "sjres"]


In [72]:
for congress in CONGRESSES:
    for bill_type in BILL_TYPES:
        # making a folder like data/raw/118/hr
        folder = f"../data/raw/{congress}/{bill_type}"
        os.makedirs(folder, exist_ok=True)


        offset = 0
        while True:
            filename = f"{folder}/page_{offset}.json"

            # skip already downloaded pages
            if os.path.exists(filename):
                with open(filename) as f:
                    data = json.load(f)
            else:
                url = f"https://api.congress.gov/v3/bill/{congress}/{bill_type}"
                params = {"offset": offset, "limit": 250, "format": "json"}
                response = requests.get(url, params=params, headers={"X-API-Key": os.environ["CONGRESS_API_KEY"]})

                response.raise_for_status()
                data = response.json()

                with open(filename, "w") as f:
                    json.dump(data, f)
                time.sleep(1) # respect rate limit

            print(congress, bill_type, "offset", offset)

            # stop when no next page
            if "next" not in data["pagination"]:
                break
            offset += 250 # increase offset to get to next page
                



118 hr offset 0
118 hr offset 250
118 hr offset 500
118 hr offset 750
118 hr offset 1000
118 hr offset 1250
118 hr offset 1500
118 hr offset 1750
118 hr offset 2000
118 hr offset 2250
118 hr offset 2500
118 hr offset 2750
118 hr offset 3000
118 hr offset 3250
118 hr offset 3500
118 hr offset 3750
118 hr offset 4000
118 hr offset 4250
118 hr offset 4500
118 hr offset 4750
118 hr offset 5000
118 hr offset 5250
118 hr offset 5500
118 hr offset 5750
118 hr offset 6000
118 hr offset 6250
118 hr offset 6500
118 hr offset 6750
118 hr offset 7000
118 hr offset 7250
118 hr offset 7500
118 hr offset 7750
118 hr offset 8000
118 hr offset 8250
118 hr offset 8500
118 hr offset 8750
118 hr offset 9000
118 hr offset 9250
118 hr offset 9500
118 hr offset 9750
118 hr offset 10000
118 hr offset 10250
118 hr offset 10500
118 s offset 0
118 s offset 250
118 s offset 500
118 s offset 750
118 s offset 1000
118 s offset 1250
118 s offset 1500
118 s offset 1750
118 s offset 2000
118 s offset 2250
118 s offset

combining bill jsons into a list that i will turn into a dataframe

In [73]:
all_bills = []
for file in Path("../data/raw").rglob("page_*.json"):
    with open(file) as f:
        data = json.load(f)
        all_bills.extend(data["bills"]) #.extend to add list elements to end of current list

df = pd.DataFrame(all_bills)

Problem: latest action is still a dict, needs broken down into separate columns somehow

In [85]:
df.head(1)

,congress,introducedDate,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,label_latest_action_date,label_latest_action_text,label_latest_action_time,bill_id
0,118,2023-01-09,8,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-12-05,2025-12-05,https://api.congress.gov/v3/bill/118/hjres/8?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN,118HJRES8


In [75]:
# using json.normalize to break latest action into separate columns
latest_action = pd.json_normalize(df["latestAction"])
df = pd.concat([df, latest_action], axis=1)
#df.drop(columns=["latestAction"], inplace=True)

In [76]:
df.rename(columns={"actionDate": "label_latest_action_date", "text": "label_latest_action_text", "actionTime": "label_latest_action_time"}, inplace=True)
df.drop(columns=["latestAction"], inplace=True)

In [77]:
df.head(5)

,congress,introducedDate,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,label_latest_action_date,label_latest_action_text,label_latest_action_time
0,118,2023-01-09,8,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-12-05,2025-12-05,https://api.congress.gov/v3/bill/118/hjres/8?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN
1,118,2023-01-09,9,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-05-27,2025-05-27,https://api.congress.gov/v3/bill/118/hjres/9?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN
2,118,2023-01-09,13,House,H,Proposing an amendment to the Constitution of ...,HJRES,2024-09-20,2024-09-20,https://api.congress.gov/v3/bill/118/hjres/13?...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN
3,118,2023-01-09,4,House,H,Proposing an amendment to the Constitution of ...,HJRES,2024-07-24,2024-07-24,https://api.congress.gov/v3/bill/118/hjres/4?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN
4,118,2023-01-09,5,House,H,Proposing an amendment to the Constitution of ...,HJRES,2024-07-24,2024-07-24,https://api.congress.gov/v3/bill/118/hjres/5?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN


Congress, type, and number will be the primary key. I'm going to make a combined column with all of them to act as ID.

In [78]:
df['bill_id'] = df['congress'].astype(str) + df['type'] + df['number'].astype(str)
df.head(1)

,congress,introducedDate,number,originChamber,originChamberCode,title,type,updateDate,updateDateIncludingText,url,label_latest_action_date,label_latest_action_text,label_latest_action_time,bill_id
0,118,2023-01-09,8,House,H,Proposing an amendment to the Constitution of ...,HJRES,2025-12-05,2025-12-05,https://api.congress.gov/v3/bill/118/hjres/8?f...,2023-01-09,Referred to the House Committee on the Judiciary.,NaN,118HJRES8


# Data Validation Checks

In [79]:
df.dtypes

congress                    int64
introducedDate                str
number                        str
originChamber                 str
originChamberCode             str
title                         str
type                          str
updateDate                    str
updateDateIncludingText       str
url                           str
label_latest_action_date      str
label_latest_action_text      str
label_latest_action_time      str
bill_id                       str
dtype: object

In [80]:
df['label_latest_action_time'].isna().sum()
# not a worthwhile column with this many NAs


np.int64(16307)

In [81]:
df.shape

(16565, 14)

In [82]:
df.groupby('type').size()


type
HJRES      230
HR       10564
S         5649
SJRES      122
dtype: int64

In [83]:
df['congress'].value_counts()

congress
118    16565
Name: count, dtype: int64

# saving data

In [86]:
folder = f"../data/processed/"
os.makedirs(folder, exist_ok=True)
df.to_csv(f"{folder}/bills.csv", index=False)

# voteview data

In [ ]:
ideology_df = pd.read_csv("../data/ideology_data/congress_118_ideology.csv")

In [ ]:
ideology_df.head(5)

,congress,chamber,icpsr,state_icpsr,district_code,state_abbrev,party_code,occupancy,last_means,bioname,...,died,nominate_dim1,nominate_dim2,nominate_log_likelihood,nominate_geo_mean_probability,nominate_number_of_votes,nominate_number_of_errors,conditional,nokken_poole_dim1,nokken_poole_dim2
0,118,President,99913,99,0,USA,100,0.0,0.0,"BIDEN, Joseph Robinette (Joe), Jr.",...,NaN,-0.320,0.283,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,118,House,20301,41,3,AL,200,NaN,NaN,"ROGERS, Mike Dennis",...,NaN,0.379,0.377,-161.82917,0.86226,1092.0,79.0,NaN,0.440,0.312
2,118,House,21102,41,7,AL,100,NaN,NaN,"SEWELL, Terri",...,NaN,-0.402,0.397,-53.66985,0.95022,1051.0,19.0,NaN,-0.522,0.361
3,118,House,21500,41,6,AL,200,NaN,NaN,"PALMER, Gary James",...,NaN,0.670,0.113,-94.35488,0.91780,1100.0,40.0,NaN,0.675,0.136
4,118,House,22108,41,1,AL,200,NaN,NaN,"CARL, Jerry L.",...,NaN,0.550,0.239,-126.19488,0.89299,1115.0,58.0,NaN,0.594,0.236


In [18]:
les_house_df = pd.read_excel("../data/les_data/house_les_all.xlsx")
les_senate_df = pd.read_excel("../data/les_data/senate_les_all.xls")

In [19]:
les_house_df.tail(5)

,Numeric counter for order in THOMAS data from 93rd - 110th Congresses,"Legislator name, as given in THOMAS","ICPSR number, according to Poole and Rosenthal",Congress number,Year at start of Congress,Two-letter state code,Congressional district number,1 = Democrat,Year first elected to House,1 = female,...,Total number of standalone bills passing House,Total number of bills receiving action in committee.1,Total number of standalone bills receiving action beyond committee,Total number of standalone bills becoming law,"Benchmark score (Classic) based on majority, seniority, chairs",LES/benchmark (Classic),"1 = Below, 2 = Meets, 3 = Above, Based on Classic LES",Lagged LES 1.0,Lagged LES 2.0,LES 2.0 Rank within Party
11601,11597.0,"Wilson, Joe",20138.0,118,2023,SC,2.0,0,2001,0,...,2,3,3,0,1.503360,0.787330,2,0.762526,1.012054,120.0
11602,11598.0,"Wittman, Robert",20756.0,118,2023,VA,1.0,0,2007,0,...,2,4,3,0,1.496394,0.732666,2,0.263959,0.932590,129.0
11603,11599.0,"Womack, Steve",21108.0,118,2023,AR,3.0,0,2010,0,...,0,1,2,0,1.491749,0.210203,1,0.047136,0.381451,134.0
11604,11600.0,"Yakym, Rudy",22171.0,118,2023,IN,2.0,0,2022,0,...,0,4,2,0,1.217968,0.408919,1,NaN,NaN,193.0
11605,11601.0,"Zinke, Ryan",21532.0,118,2023,MT,1.0,0,2022,0,...,0,7,1,0,1.217968,0.483651,1,NaN,NaN,180.0


In [7]:
les_senate_df.head(5)

,last name,first name,two letter state abbreviation,congress number,Observation number order based on scraping congress.gov,icpsr number,year elected for congress,1 if senator is democrat,1 if senator is in majority party,year first elected to the senate,...,Total number of standalone bills passing Senate,Total number of standalone bills becoming law,"Benchmark score (classic) based on majority, seniority, chairs",LES/Benchmark (classic),"1 = Below, 2 = Meets, 3 = Above, Based on Classic LES",Lagged LES 1.0,Lagged LES 2.0,Number of members in Party (0 = Indep),LES Classic Rank within Party,LES 2.0 Rank within Party
0,Abourezk,James,SD,93,1,13000,1972,1,1,1972.0,...,1,0,0.420971,0.416148,1,NaN,NaN,57,54,0
1,Aiken,George,VT,93,2,52,1972,0,0,1940.0,...,3,1,1.028909,0.395388,1,NaN,NaN,42,23,0
2,Allen,James,AL,93,3,12100,1972,1,1,1968.0,...,1,0,0.942945,0.227074,1,NaN,NaN,57,53,0
3,Baker,Howard,TN,93,4,11200,1972,0,0,1966.0,...,5,2,0.461136,2.027374,3,NaN,NaN,42,6,0
4,Bartlett,Dewey,OK,93,5,14100,1972,0,0,1972.0,...,6,3,0.330112,1.995912,3,NaN,NaN,42,12,0


In [16]:
les_house_df = les_house_df.rename(columns={'congress number': 'congress'}, inplace=True)

#les_house_df = les_house_df[les_house_df['congress'] == 118]

Notes for next time I look at this: 
- congress seems like it will work as a joining key for all three data sets once I rename it in the les_house_df and les_senate_df. currently called `congress number` in the Center for Effective Lawmaking data
- les_house_df and les_senate_df need joined
- likely going to have to use LES 1.0 for all
- i'm not completely sure how to join the bill data with the member's of congress data or if we have to? not thinking very well atm
- 